#### Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer

In [2]:
df = pd.read_csv("cleaned_data.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      4803 non-null   int64
 1   title   4803 non-null   str  
 2   tags    4803 non-null   str  
dtypes: int64(1), str(2)
memory usage: 5.7 MB


me, but it also takes into account how important a word is in the entire dataset. 
It gives less weight to common words that appear in many documents and more weight to words that are unique to a few documents.
CountVectorizer/TfidfVectorizer is a tool in scikit-learn that converts text data into numbers.

CountVectorizer counts how many times each word appears in each document.
TfidfVectorizer does the sa
When you use CountVectorizer or TfidfVectorizer, you can specify parameters like max_features to limit the number of features (words) to keep. In this case, we want to keep only the top 10,000 most frequent words from the dataset.
Use only the top 10,000 most frequent words from the dataset. i.e (all movies x 10000 unique words) shaped matrix 
Remove common English words that do not add meaning.

When .fit_transform() runs, CountVectorizer calls an internal, built-in python function called a tokenizer. 
By default, it uses a regular expression (r"(?u)\b\w\w+\b") to split your long string of tags into individual words of 2 or more characters.
You didn't have to write code for it because scikit-learn did the heavy lifting automatically behind the scenes.
 
E.g.
lets say vectorizer finds 35,000 unique words.
Instead of keeping all 35,000, it sorts them by their term frequency (how many times they appear across the whole dataset) from highest to lowest. 
It then slices off the top 10,000 most frequent words and throws away the rest.

In [4]:
# ------------------------------------ TF-IDF ------------------------------------

# TF-IDF prioritizes unique plot elements and specific actor/director names over generic tags,
# Your Recommendations will feel much more accurate and "smart" to the end user.

tfidf = TfidfVectorizer(max_features = 10000, stop_words='english')
vector = tfidf.fit_transform(df['tags']).toarray()          

In [5]:
vector.shape

(4803, 10000)

In [6]:
similarity = cosine_similarity(vector)
similarity

array([[1.        , 0.08909061, 0.13541703, ..., 0.03460067, 0.02031206,
        0.02923523],
       [0.08909061, 1.        , 0.05273835, ..., 0.01895012, 0.01186052,
        0.01511501],
       [0.13541703, 0.05273835, 1.        , ..., 0.03169707, 0.030465  ,
        0.00841669],
       ...,
       [0.03460067, 0.01895012, 0.03169707, ..., 1.        , 0.02995468,
        0.0280435 ],
       [0.02031206, 0.01186052, 0.030465  , ..., 0.02995468, 1.        ,
        0.01383299],
       [0.02923523, 0.01511501, 0.00841669, ..., 0.0280435 , 0.01383299,
        1.        ]], shape=(4803, 4803))

In [7]:
similarity.shape

(4803, 4803)

In [8]:
#Lets write function get name of movie by index
def get_name_by_index(i):
    if i < len(df) and i>0:
        return df.loc[i,'title']
    else:
        return''

In [9]:
a = get_name_by_index(10)
a

'Superman Returns'

In [10]:
# # Function to improvise the logic for getting the name.
# # if user enter the spider man it will give error as in real dataset name is like spider-man.
# # so it will not match the movie name. it will show movie not found. so we need to handle spaces and -  in this case.

# def get_index_from_name(name):
#     # Normalize user input: lowercase it and strip out all spaces and hyphens
#     clean_user_name = name.strip().lower().replace(' ', '').replace('-', '')
    
#     # Vectorized pandas match: normalize the dataframe column for comparison
#     match = df[df['title'].str.lower().str.replace(' ', '').str.replace('-', '') == clean_user_name]
    
#     if not match.empty:
#         return match.index[0]
#     return -1

# #Lets write function get index of movie by movie name

# def get_index_from_name(name):
#     found_index = -1
#     for i in df.index:
#         if df.loc[i,'title'].strip().lower() == name.strip().lower():
#             found_index = i
#             break
#     return found_index

In [11]:
#b = get_index_from_name("Spider-man 3")
#b

In [12]:
# name = input("Enter movie you watched:")
# index = get_index_from_name(name)
# if index != -1:
#     similarity_indexes =  list(enumerate(similarity[index]))
#     similarity_indexes = sorted(similarity_indexes, key=lambda x: x[1], reverse=True)
#     print("You must watch following movies:")
#     for i in range(1, 6):
#         print(i, ":", get_name_by_index(similarity_indexes[i][0]))
# else:
#     print("Movie Not Found")    
    
    

In [13]:
# Function to improvise the logic for getting the name.
# if user enter the spider man it will give error as in real dataset name is like spider-man.
# so it will not match the movie name. it will show movie not found. so we need to handle spaces and -  in this case.

def get_index_from_name(name):
    # Normalize user input: lowercase it and strip out all spaces and hyphens
    clean_user_name = name.strip().lower().replace(' ', '').replace('-', '')
    
    # Vectorized pandas match: normalize the dataframe column for comparison
    match = df[df['title'].str.lower().str.replace(' ', '').str.replace('-', '') == clean_user_name]
    # spider man 3 == spider man 3
    if not match.empty:
        return match.index[0]
    return -1

In [14]:
get_index_from_name("spider man 3")

5

In [ ]:
name = input("Enter movie you watched:")
index = get_index_from_name(name)
# similarity_indexes = list(enumerate(similarity[index]))
# similarity_indexes = sorted(similarity_indexes, key=lambda x: x[1], reverse=True)
# similarity_indexes

if index != -1:
    # Fetch similarity rankings. enumerate() pairs each score with its original movie index like (5, np.float64(0.3635931236853142))
    similarity_indexes = list(enumerate(similarity[index]))

    # The very first item in this sorted list (index 0) will always be the movie itself with a perfect similarity score of 1.0
    similarity_indexes = sorted(similarity_indexes, key=lambda x: x[1], reverse=True)
    
    print(f"\nBecause you watched '{df.loc[index, 'title']}':")
    print("You must watch the following movies:")
    
    # Loop to print top 5 recommendations (skipping index 0 as it's the movie itself)
    for i in range(1, 6):
        movie_idx = similarity_indexes[i][0]
        print(i, ":", get_name_by_index(movie_idx))
else:
    print("Movie Not Found")

In [ ]:
#Lets export similarity 
import pickle as pkl
pkl.dump(similarity, open('similarity.pkl', 'wb'))